# HDMI GUI viewer (single-cell)

Brings up the integrated AudioLab HDMI path (`axi_vdma_hdmi` + `v_tc_hdmi`) and scans out the compact-v2 800x480 GUI on the 5-inch LCD.

While the animation loop runs, the cell prints a live resource report (CPU%, RAM, render/write timings, achieved FPS, VDMA error bits, current offset). Stop with Jupyter `Interrupt Kernel` (`I, I`); the `finally` block stops VDMA cleanly.

## Calibration

`OFFSET_X` / `OFFSET_Y` shift the 800x480 logical GUI inside the fixed 1280x720 HDMI framebuffer to compensate when the LCD's visible viewport is **not** flush against framebuffer `(0,0)`. The Phase 5C default is `(0, 0)`; raise these values until the GUI is flush against the LCD's top-left edge. Negative values are accepted (the renderer crops the corresponding left/top strip).

## Constraints

See `CLAUDE.md`: `AudioLabOverlay()` is loaded once, no `Overlay("base.bit")`, no `run_pynq_hdmi()`, framebuffer is the fixed 1280x720 HDMI signal.

In [ ]:
# =====================================================================
# CONFIG -- tweak these if the LCD viewport is shifted relative to (0,0)
# =====================================================================
OFFSET_X = 0          # framebuffer X where the 800x480 GUI is placed
OFFSET_Y = 0          # framebuffer Y where the 800x480 GUI is placed
TARGET_FPS = 5
REPORT_EVERY = 1.0    # seconds between resource reports
# =====================================================================

import sys, time, math, json, os
from IPython.display import clear_output, display
import ipywidgets as widgets

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for path in (PROJECT_ROOT, PROJECT_ROOT + "/GUI"):
    if path not in sys.path:
        sys.path.insert(0, path)

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.hdmi_backend import AudioLabHdmiBackend
from pynq_multi_fx_gui import (
    AppState, load_state_json, make_pynq_static_render_cache,
    render_frame_800x480_compact_v2,
    DIST_MODELS, AMP_MODELS, CAB_MODELS,
)

# ---- /proc-based resource samplers (no psutil dependency) ----------
def _read_cpu_times():
    with open("/proc/stat") as fp:
        fields = fp.readline().split()[1:]
    vals = [int(v) for v in fields]
    idle = vals[3] + (vals[4] if len(vals) > 4 else 0)
    return sum(vals), idle

def _cpu_percent(prev):
    total, idle = _read_cpu_times()
    dt = total - prev[0]
    di = idle - prev[1]
    pct = 100.0 * (1.0 - di / dt) if dt > 0 else 0.0
    return pct, (total, idle)

def _read_meminfo():
    info = {}
    with open("/proc/meminfo") as fp:
        for line in fp:
            k, _, rest = line.partition(":")
            info[k.strip()] = int(rest.split()[0])  # kB
    total_kb = info.get("MemTotal", 0)
    avail_kb = info.get("MemAvailable", info.get("MemFree", 0))
    used_kb = max(total_kb - avail_kb, 0)
    return {
        "total_mb": total_kb / 1024.0,
        "used_mb":  used_kb / 1024.0,
        "pct":      (100.0 * used_kb / total_kb) if total_kb else 0.0,
    }

def _read_loadavg():
    with open("/proc/loadavg") as fp:
        parts = fp.read().split()
    return tuple(float(p) for p in parts[:3])

# ---- overlay + smoke ----------------------------------------------
overlay = AudioLabOverlay()
ip_keys = set(getattr(overlay, "ip_dict", {}).keys())
smoke = {
    "ADC HPF": bool(overlay.codec.get_adc_hpf_state()),
    "R19": "0x{:02x}".format(int(overlay.codec.R19_ADC_CONTROL[0]) & 0xFF),
    "axi_vdma_hdmi": "axi_vdma_hdmi" in ip_keys,
    "v_tc_hdmi":     "v_tc_hdmi" in ip_keys,
}
assert smoke["axi_vdma_hdmi"] and smoke["v_tc_hdmi"], ("HDMI IPs missing", smoke)
assert smoke["R19"] == "0x23", ("ADC HPF must be default-on", smoke)
print("smoke:", json.dumps(smoke, sort_keys=True))

# ---- initial frame + HDMI start -----------------------------------
try:
    state = load_state_json()
except Exception:
    state = AppState()
cache = make_pynq_static_render_cache()
frame = render_frame_800x480_compact_v2(state, cache=cache)
assert frame.shape == (480, 800, 3) and str(frame.dtype) == "uint8"

backend = AudioLabHdmiBackend(overlay)
backend.start(frame, placement="manual",
              offset_x=OFFSET_X, offset_y=OFFSET_Y)
time.sleep(0.1)
status0 = backend.status()
errs0 = backend.errors()
last0 = status0.get("last_frame_write", {}) or {}
fb_region = last0.get("framebuffer_copied_region", {})
assert not (errs0.get("dmainterr") or errs0.get("dmaslverr") or errs0.get("dmadecerr")), errs0
print("framebuffer region: x={x0}..{x1} y={y0}..{y1}".format(**fb_region))
print("clipped:", last0.get("clipped"), " negative_offset:", last0.get("negative_offset"))
print("hdmi backend started; entering live loop (Interrupt Kernel to stop)")

# ---- DIST / AMP / CAB model dropdowns -----------------------------
# These dropdowns drive ONLY the HDMI GUI state used by the
# compact-v2 renderer. They intentionally do NOT call
# overlay.set_amp_model(...) / overlay.set_guitar_effects(...) here,
# because set_guitar_effects() resets every other effect section to
# its default kwargs and would wipe out the user's current voicing.
# DSP-side model switching is the job of GuitarPedalboardOneCell.ipynb,
# which always writes a full-state set_guitar_effects(**kwargs) call.
def _clamp_model_idx(value, length):
    try:
        value = int(value)
    except (TypeError, ValueError):
        value = 0
    if value < 0:
        return 0
    if value >= length:
        return length - 1
    return value

def _dist_model_options():
    return [(name, i) for i, name in enumerate(DIST_MODELS)]

def _amp_model_options():
    return [(item[0], i) for i, item in enumerate(AMP_MODELS)]

def _cab_model_options():
    # Three DSP-supported cab models live at indices 0..2 in CAB_MODELS;
    # indices 3+ have no live Clash stage (audio_lab_gui_bridge.py
    # clamps cab_model to 0..2 before writing the GPIO word). Only
    # offer the three real models in the HDMI GUI dropdown so the
    # selection always corresponds to something the renderer can
    # actually reflect.
    return [(name, i) for i, name in enumerate(CAB_MODELS[:3])]

w_dist_model = widgets.Dropdown(
    options=_dist_model_options(),
    value=_clamp_model_idx(getattr(state, "dist_model_idx", 0), len(DIST_MODELS)),
    description="DIST model:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="320px"),
)
w_amp_model = widgets.Dropdown(
    options=_amp_model_options(),
    value=_clamp_model_idx(getattr(state, "amp_model_idx", 0), len(AMP_MODELS)),
    description="AMP model:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="320px"),
)
w_cab_model = widgets.Dropdown(
    options=_cab_model_options(),
    value=_clamp_model_idx(getattr(state, "cab_model_idx", 0), 3),
    description="CAB model:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="320px"),
)

def _on_dist_model(change):
    state.dist_model_idx = _clamp_model_idx(change["new"], len(DIST_MODELS))
def _on_amp_model(change):
    state.amp_model_idx = _clamp_model_idx(change["new"], len(AMP_MODELS))
def _on_cab_model(change):
    state.cab_model_idx = _clamp_model_idx(change["new"], 3)

w_dist_model.observe(_on_dist_model, names="value")
w_amp_model.observe(_on_amp_model, names="value")
w_cab_model.observe(_on_cab_model, names="value")

status_out = widgets.Output()
display(widgets.VBox([
    widgets.HBox([w_dist_model, w_amp_model, w_cab_model]),
    widgets.HTML(
        "<small>Dropdowns update only the on-screen MODEL row in the "
        "HDMI GUI. For actual DSP-side model switching use "
        "<code>GuitarPedalboardOneCell.ipynb</code>.</small>"),
    status_out,
]))

# ---- live loop + resource monitor ---------------------------------
dt = 1.0 / TARGET_FPS
prev_cpu = _read_cpu_times()
win_t0 = time.time()
frames = 0
render_ms_sum = 0.0
write_ms_sum  = 0.0
render_ms_max = 0.0
write_ms_max  = 0.0
start_wall = time.time()
total_frames = 0

try:
    while True:
        state.t += dt
        state.in_level  = 0.55 + 0.18 * math.sin(state.t * 1.3)
        state.out_level = 0.62 + 0.16 * math.sin(state.t * 1.5 + 0.7)

        t0 = time.time()
        frame = render_frame_800x480_compact_v2(state, cache=cache)
        t1 = time.time()
        backend.write_logical_frame(frame, placement="manual",
                                    offset_x=OFFSET_X, offset_y=OFFSET_Y)
        t2 = time.time()

        rm = (t1 - t0) * 1000.0
        wm = (t2 - t1) * 1000.0
        render_ms_sum += rm
        write_ms_sum  += wm
        if rm > render_ms_max: render_ms_max = rm
        if wm > write_ms_max:  write_ms_max  = wm
        frames += 1
        total_frames += 1

        win_dt = time.time() - win_t0
        if win_dt >= REPORT_EVERY:
            cpu_pct, prev_cpu = _cpu_percent(prev_cpu)
            mem = _read_meminfo()
            la1, la5, la15 = _read_loadavg()
            errs = backend.errors()
            last = backend.status().get("last_frame_write", {}) or {}
            fb = last.get("framebuffer_copied_region", {})
            fps  = frames / win_dt
            avg_render = render_ms_sum / frames
            avg_write  = write_ms_sum  / frames

            status_out.clear_output(wait=True)
            with status_out:
                print("=== HDMI GUI live (compact-v2 800x480) ===")
                print("offset          : x={} y={}  (framebuffer 1280x720)".format(
                    OFFSET_X, OFFSET_Y))
                print("framebuffer rect: x={x0}..{x1} y={y0}..{y1}  clipped={clip}".format(
                    clip=last.get("clipped"),
                    **{k: fb.get(k, "?") for k in ("x0","x1","y0","y1")}))
                print("uptime          : {:.1f} s   total frames: {}".format(
                    time.time() - start_wall, total_frames))
                print("target / actual : {} fps / {:.2f} fps  (last {:.2f} s window)".format(
                    TARGET_FPS, fps, win_dt))
                print("render time     : avg {:.2f} ms   max {:.2f} ms".format(
                    avg_render, render_ms_max))
                print("vdma write time : avg {:.2f} ms   max {:.2f} ms".format(
                    avg_write, write_ms_max))
                print("cpu             : {:5.1f} %    loadavg: {:.2f} {:.2f} {:.2f}".format(
                    cpu_pct, la1, la5, la15))
                print("memory          : {:.0f} / {:.0f} MB ({:.1f} %)".format(
                    mem["used_mb"], mem["total_mb"], mem["pct"]))
                print("vdma errors     : {}".format(errs))
                print("\n(Interrupt Kernel `I, I` to stop; backend.stop() runs automatically)")
                print("(Adjust OFFSET_X / OFFSET_Y at the top of this cell and re-run if the LCD is shifted)")

            win_t0 = time.time()
            frames = 0
            render_ms_sum = 0.0
            write_ms_sum  = 0.0
            render_ms_max = 0.0
            write_ms_max  = 0.0

        sleep_s = dt - (time.time() - t0)
        if sleep_s > 0:
            time.sleep(sleep_s)
except KeyboardInterrupt:
    print("\ninterrupt received, stopping...")
finally:
    try:
        backend.stop()
        print("hdmi backend stopped cleanly")
    except Exception as exc:
        print("backend.stop() raised:", exc)